# Cohort Configuration

Shared config notebook for the 6 patient journey validation notebooks. 
Defines the offset, data boundaries, cohort labels (C1–C22), and the cohort assignment function.

**Usage:** Each patient journey notebook should `%run` this notebook early on.

In [0]:
# ============================================================
# GLOBAL PARAMETERS — change these to configure the pipeline
# ============================================================

OFFSET_DAY = 7            # Day-of-month offset (0 = standard calendar months)
DATA_START_DATE = "2023-01-01"   # Exclude all data before this date
COHORT_ORIGIN  = "2025-01-01"    # Reference month of C1
COHORT_LAST    = "2026-10-01"    # Reference month of C22 (last cohort)

print(f"OFFSET_DAY      = {OFFSET_DAY}")
print(f"DATA_START_DATE = {DATA_START_DATE}")
print(f"COHORT_ORIGIN   = {COHORT_ORIGIN}")
print(f"COHORT_LAST     = {COHORT_LAST}")

OFFSET_DAY      = 7
DATA_START_DATE = 2023-01-01
COHORT_ORIGIN   = 2025-01-01
COHORT_LAST     = 2026-10-01


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

def assign_cohort_month(svc_dt_col):
    """
    Maps a service-date column to its cohort reference month.

    With OFFSET_DAY = 0  → standard trunc to calendar month.
    With OFFSET_DAY = 7  → cohort boundaries fall on the 7th:
        C1 = [Jan 7, Feb 7),  C2 = [Feb 7, Mar 7), …

    The 'offset-th' day belongs to the NEW cohort (start-inclusive).
    """
    if OFFSET_DAY == 0:
        return F.trunc(svc_dt_col, "month")
    else:
        return F.trunc(F.date_sub(svc_dt_col, OFFSET_DAY - 1), "month")


def _cohort_month_sql(col_name):
    """
    Returns the SQL expression string for assign_cohort_month.
    Use inside F.expr() where a PySpark Column cannot be used (e.g. sequence()).
    """
    if OFFSET_DAY == 0:
        return f"trunc({col_name}, 'month')"
    else:
        return f"trunc(date_sub({col_name}, {OFFSET_DAY - 1}), 'month')"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType
from dateutil.relativedelta import relativedelta
from datetime import date, timedelta

# Build the cohort lookup: C1 through C22
_origin = date.fromisoformat(COHORT_ORIGIN)
_last   = date.fromisoformat(COHORT_LAST)

cohort_rows = []
_current = _origin
_idx = 1
while _current <= _last:
    _next = _current + relativedelta(months=1)
    # Actual calendar boundaries with offset
    if OFFSET_DAY == 0:
        cohort_start = _current
        cohort_end   = _next
    else:
        cohort_start = _current.replace(day=OFFSET_DAY)
        cohort_end   = _next.replace(day=OFFSET_DAY)
    cohort_rows.append((f"C{_idx}", _current, cohort_start, cohort_end))
    _current = _next
    _idx += 1

schema = StructType([
    StructField("cohort_label",      StringType(), False),
    StructField("cohort_ref_month",  DateType(),   False),  # the reference month (Jan 1, Feb 1, …)
    StructField("cohort_start",      DateType(),   False),  # actual start date with offset
    StructField("cohort_end",        DateType(),   False),  # actual end date (exclusive)
])

df_cohort_ref = spark.createDataFrame(cohort_rows, schema=schema)
df_cohort_ref.display()
print(f"\n{len(cohort_rows)} cohorts defined (C1 – C{len(cohort_rows)})")

cohort_label,cohort_ref_month,cohort_start,cohort_end
C1,2025-01-01,2025-01-07,2025-02-07
C2,2025-02-01,2025-02-07,2025-03-07
C3,2025-03-01,2025-03-07,2025-04-07
C4,2025-04-01,2025-04-07,2025-05-07
C5,2025-05-01,2025-05-07,2025-06-07
C6,2025-06-01,2025-06-07,2025-07-07
C7,2025-07-01,2025-07-07,2025-08-07
C8,2025-08-01,2025-08-07,2025-09-07
C9,2025-09-01,2025-09-07,2025-10-07
C10,2025-10-01,2025-10-07,2025-11-07



22 cohorts defined (C1 – C22)


In [0]:
# ============================================================
# VALIDATION FILTER — 10 test HCPs (shared across all 6 notebooks)
# Remove this block for production runs.
# ============================================================

TEST_NPIS = [
    "1396374062", "1508230566", "1801062104", "1386651560", "1528629052",
    "1659887388", "1215675616", "1467950774", "1235697863", "1841364841"
]

def apply_hcp_filter(df, npi_col="NPI_NUMBER", patient_col="PATIENT_ID"):
    """
    Filters a DataFrame to only patients linked to the test HCPs.
    Returns (filtered_df, patient_list).
    """
    test_patients = [
        row[patient_col] for row in
        df.filter(F.col(npi_col).isin(TEST_NPIS))
          .select(patient_col).distinct().collect()
    ]
    df_filtered = df.filter(F.col(patient_col).isin(test_patients))
    print(f"Filtered to {len(test_patients)} patients across {len(TEST_NPIS)} HCPs")
    return df_filtered, test_patients

In [0]:
# Quick self-test: verify assign_cohort_month with known examples
from datetime import date as _d

_test_cases = [
    ("2025-01-01", 0,  "2025-01-01"),  # offset 0: standard month
    ("2025-01-07", 7,  "2025-01-01"),  # offset 7: Jan 7 → C1 (Jan)
    ("2025-01-06", 7,  "2024-12-01"),  # offset 7: Jan 6 → lookback (Dec)
    ("2025-02-06", 7,  "2025-01-01"),  # offset 7: Feb 6 → still C1 (Jan)
    ("2025-02-07", 7,  "2025-02-01"),  # offset 7: Feb 7 → C2 (Feb)
    ("2025-10-04", 7,  "2025-09-01"),  # offset 7: Oct 4 → C9 (Sep)
]

print(f"Testing with current OFFSET_DAY = {OFFSET_DAY}")
print("-" * 55)
for svc, off, expected in _test_cases:
    if off != OFFSET_DAY:
        continue
    test_df = spark.createDataFrame([(svc,)], ["SVC_DT"]).withColumn("SVC_DT", F.to_date("SVC_DT"))
    result = test_df.select(assign_cohort_month(col("SVC_DT")).alias("cohort_month")).collect()[0][0]
    status = "PASS" if str(result) == expected else "FAIL"
    print(f"  {svc} → {result}  (expected {expected})  [{status}]")

print("\nConfig notebook ready. All 6 patient journey notebooks can %run this.")

Testing with current OFFSET_DAY = 7
-------------------------------------------------------
  2025-01-07 → 2025-01-01  (expected 2025-01-01)  [PASS]
  2025-01-06 → 2024-12-01  (expected 2024-12-01)  [PASS]
  2025-02-06 → 2025-01-01  (expected 2025-01-01)  [PASS]
  2025-02-07 → 2025-02-01  (expected 2025-02-01)  [PASS]
  2025-10-04 → 2025-09-01  (expected 2025-09-01)  [PASS]

Config notebook ready. All 6 patient journey notebooks can %run this.
